In [1]:
print("Check_Python")
%load_ext autotime

Check_Python
time: 0 ns (started: 2026-10-02 21:27:30 +05:30)


In [2]:
import tensorflow as tf
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import random

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import layers, models, regularizers

from sklearn.utils.class_weight import compute_class_weight

time: 24.8 s (started: 2026-10-02 21:27:30 +05:30)


In [3]:
print(tf.__version__)
print(tf.config.list_physical_devices())

2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
time: 0 ns (started: 2026-10-02 21:27:55 +05:30)


In [4]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

MODELS_DIR = PROJECT_ROOT / "models" / "phase1"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(TRAIN_DIR.exists(), VAL_DIR.exists(), TEST_DIR.exists(), MODELS_DIR.exists())

True True True True
time: 0 ns (started: 2026-10-02 21:27:55 +05:30)


In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"
SEED = 42

time: 0 ns (started: 2026-10-02 21:27:55 +05:30)


In [6]:
CLASSES = ["NORMAL", "PNEUMONIA"]

time: 15 ms (started: 2026-10-02 21:27:55 +05:30)


In [7]:
bad_files = (
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpeg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.png"))
)

deleted = 0

for f in bad_files:
    try:
        f.unlink()
        deleted += 1
    except Exception as e:
        print(f"Error deleting {f}: {e}")

print(f"Deleted {deleted} augmented files")

Deleted 0 augmented files
time: 16 ms (started: 2026-10-02 21:27:55 +05:30)


In [8]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

time: 0 ns (started: 2026-10-02 21:27:55 +05:30)


Critical nuance — the rescale here does NOT apply during training. The custom generator (make_balanced_fold_gen) calls datagen.random_transform(), which applies only the geometric transforms. rescale is only applied by Keras's own flow_* methods — not by random_transform. The rescaling in this notebook is instead done manually (img / 255.0) inside the generator. So rescale=1/255 inside train_datagen is redundant but harmless — it simply has no effect.

In [9]:
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

time: 0 ns (started: 2026-10-02 21:27:55 +05:30)


In [10]:
def make_balanced_fold_gen(paths, labels, batch_size=32, datagen=None):
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]

    while True:
        batch_paths, batch_labels = [], []

        for _ in range(batch_size // 2):
            p = np.random.choice(pos_idx)
            n = np.random.choice(neg_idx)

            batch_paths += [paths[p], paths[n]]
            batch_labels += [1, 0]

        combined = list(zip(batch_paths, batch_labels))
        random.shuffle(combined)
        batch_paths, batch_labels = zip(*combined)

        images = []
        for path in batch_paths:
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (224, 224))
            img = img / 255.0
        
            img = np.expand_dims(img, axis=-1)  
        
            if datagen is not None:
                img = datagen.random_transform(img)  
        
            images.append(img)

        yield np.array(images), np.array(batch_labels)

time: 15 ms (started: 2026-10-02 21:27:55 +05:30)


In [11]:
label_to_classname = {0: "NORMAL", 1: "PNEUMONIA"}

time: 15 ms (started: 2026-10-02 21:27:55 +05:30)


In [12]:
def make_val_generator(val_paths, val_labels):

    val_df = pd.DataFrame({
        "filename": val_paths,
        "class": [label_to_classname[l] for l in val_labels]
    })

    return val_test_datagen.flow_from_dataframe(
        val_df,
        x_col="filename",
        y_col="class",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="binary",
        color_mode=COLOR_MODE,
        shuffle=False
    )

time: 16 ms (started: 2026-10-02 21:27:56 +05:30)


In [13]:
def build_model():
    m = models.Sequential([
        layers.Input(shape=(224, 224, 1)),

        layers.Conv2D(32, (3,3), padding='same',
                      kernel_regularizer=regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.1),

        layers.Conv2D(64, (3,3), padding='same',
                      kernel_regularizer=regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(1, activation='sigmoid')
    ])

    m.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )

    return m

time: 0 ns (started: 2026-10-02 21:27:56 +05:30)


Execution order reminder: The below cell must run after the deletion cell. If you re-run the notebook top-to-bottom, this is fine. If you re-run only this cell after training, deleted aug_* files won't be in all_paths either. Just be careful not to run this cell before the deletion cell.

In [14]:
VALID_EXTS = {".jpeg", ".jpg", ".png"}
 
all_paths = []
all_labels = []

temp_gen = train_datagen.flow_from_directory( #dummy generator
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=1,
    class_mode="binary",
    shuffle=False
)

class_map = temp_gen.class_indices  
 
for class_name, label in class_map.items():
    class_dir = TRAIN_DIR / class_name
 
    for img_name in os.listdir(class_dir):
        if Path(img_name).suffix.lower() not in VALID_EXTS:
            continue
        all_paths.append(str(class_dir / img_name))
        all_labels.append(label)
 
all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
 
print("Total samples:", len(all_paths))
print("Class distribution:", np.bincount(all_labels))

Found 4693 images belonging to 2 classes.
Total samples: 4693
Class distribution: [1206 3487]
time: 969 ms (started: 2026-10-02 21:27:56 +05:30)


VALID_EXTS bug confirmed fixed. Path(img_name).suffix correctly extracts the extension from a bare filename (e.g., "image.jpeg" → ".jpeg"). This works regardless of whether img_name is a full path or just a filename. The previous concern about this was unfounded — Path.suffix works on filename strings directly.

In [15]:
def custom_stratified_kfold_split(paths, labels, n_splits=3, shuffle=True, random_state=42):
    """
    Custom implementation of StratifiedKFold.split().
    Yields (train_idx, val_idx) for each fold, preserving class proportions.
    """
    rng = np.random.RandomState(random_state)   # legacy RandomState

    # Group indices by class label
    unique_classes = np.unique(labels)
    class_indices = {cls: np.where(labels == cls)[0] for cls in unique_classes}
    
    # Optionally shuffle within each class
    if shuffle:
        for cls in unique_classes:
            rng.shuffle(class_indices[cls])
    
    # Split each class's indices into n_splits roughly equal folds
    class_folds = {}
    for cls in unique_classes:
        class_folds[cls] = np.array_split(class_indices[cls], n_splits)
    
    # For each fold, val = that fold's chunk per class, train = the rest
    for fold_idx in range(n_splits):
        val_indices = []
        train_indices = []
        
        for cls in unique_classes:
            val_indices.append(class_folds[cls][fold_idx])
            train_indices.extend(
                class_folds[cls][i] for i in range(n_splits) if i != fold_idx
            )
        
        val_idx   = np.concatenate(val_indices)
        train_idx = np.concatenate(train_indices)
        
        # Shuffle the final index arrays so classes aren't in blocks
        if shuffle:
            rng.shuffle(val_idx)
            rng.shuffle(train_idx)
        
        yield train_idx, val_idx

time: 16 ms (started: 2026-10-02 21:27:57 +05:30)


In [16]:
for fold, (train_idx, val_idx) in enumerate(
    custom_stratified_kfold_split(all_paths, all_labels, n_splits=3, shuffle=True, random_state=42), 1
):
    print(f"\n------------ Fold {fold} ------------")

    train_paths_fold  = all_paths[train_idx]
    val_paths_fold    = all_paths[val_idx]
    train_labels_fold = all_labels[train_idx]
    val_labels_fold   = all_labels[val_idx]

    train_gen = make_balanced_fold_gen(
        train_paths_fold,
        train_labels_fold,
        batch_size=BATCH_SIZE,
        datagen=train_datagen
    )

    val_gen = make_val_generator(val_paths_fold, val_labels_fold)
    model   = build_model()

    steps_per_epoch = len(train_paths_fold) // BATCH_SIZE

    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
        ModelCheckpoint(
            filepath=str(MODELS_DIR / f"best_model_kfoldc_{fold}.h5"),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

    history = model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_gen,
        epochs=15,
        callbacks=callbacks
    )


------------ Fold 1 ------------
Found 1565 validated image filenames belonging to 2 classes.
Epoch 1/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6298 - loss: 3.7177 - precision: 0.6369 - recall: 0.6142
Epoch 1: val_accuracy improved from None to 0.74313, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_kfoldc_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 331s 3s/step - accuracy: 0.7287 - loss: 1.5441 - precision: 0.7415 - recall: 0.7023 - val_accuracy: 0.7431 - val_loss: 0.5348 - val_precision: 0.7431 - val_recall: 1.0000 - learning_rate: 1.0000e-04
Epoch 2/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8597 - loss: 0.3451 - precision: 0.8632 - recall: 0.8547
Epoch 2: val_accuracy did not improve from 0.74313
97/97 ━━━━━━━━━━━━━━━━━━━━ 410s 4s/step - accuracy: 0.8740 - loss: 0.3258 - precision: 0.8757 - recall: 0.8718 - val_accuracy: 0.7431 - val_loss: 0.6661 - val_precision: 0.7431 - val_recall: 1.0000 - learning_rate: 1.0000e-04
Epoch 3/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8751 - loss: 0.3140 - precision: 0.8868 - recall: 0.8602
Epoch 3: val_accuracy did not improve from 0.74313
97/97 ━━━━━━━━━━━━━━━━━━━━ 352s 4s/step - accuracy: 0.8824 - loss: 0.2991 - precision: 0.8938 - recall: 0.8679 - val_accuracy: 0.7431 - val_loss: 0.7435 - val_precision: 0.7431 - val_recall: 1.0000 - learni

97/97 ━━━━━━━━━━━━━━━━━━━━ 367s 4s/step - accuracy: 0.9265 - loss: 0.2172 - precision: 0.9321 - recall: 0.9201 - val_accuracy: 0.7597 - val_loss: 0.5479 - val_precision: 0.7557 - val_recall: 1.0000 - learning_rate: 3.0000e-05
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9427 - loss: 0.1861 - precision: 0.9438 - recall: 0.9416
Epoch 6: val_accuracy improved from 0.75974 to 0.85559, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_kfoldc_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 374s 4s/step - accuracy: 0.9278 - loss: 0.2075 - precision: 0.9329 - recall: 0.9220 - val_accuracy: 0.8556 - val_loss: 0.3135 - val_precision: 0.8383 - val_recall: 0.9983 - learning_rate: 9.0000e-06
Epoch 7/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9278 - loss: 0.2088 - precision: 0.9280 - recall: 0.9275
Epoch 7: val_accuracy improved from 0.85559 to 0.95144, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_kfoldc_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 393s 4s/step - accuracy: 0.9298 - loss: 0.2080 - precision: 0.9348 - recall: 0.9240 - val_accuracy: 0.9514 - val_loss: 0.1246 - val_precision: 0.9594 - val_recall: 0.9759 - learning_rate: 9.0000e-06
Epoch 8/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9107 - loss: 0.2180 - precision: 0.9174 - recall: 0.9029
Epoch 8: val_accuracy improved from 0.95144 to 0.95335, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_kfoldc_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 394s 4s/step - accuracy: 0.9201 - loss: 0.2111 - precision: 0.9312 - recall: 0.9072 - val_accuracy: 0.9534 - val_loss: 0.1332 - val_precision: 0.9875 - val_recall: 0.9493 - learning_rate: 9.0000e-06
Epoch 9/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9169 - loss: 0.2173 - precision: 0.9229 - recall: 0.9099
Epoch 9: val_accuracy did not improve from 0.95335
97/97 ━━━━━━━━━━━━━━━━━━━━ 393s 4s/step - accuracy: 0.9233 - loss: 0.2151 - precision: 0.9334 - recall: 0.9117 - val_accuracy: 0.9291 - val_loss: 0.1847 - val_precision: 0.9925 - val_recall: 0.9114 - learning_rate: 9.0000e-06
Epoch 10/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9253 - loss: 0.2126 - precision: 0.9369 - recall: 0.9122
Epoch 10: val_accuracy did not improve from 0.95335
97/97 ━━━━━━━━━━━━━━━━━━━━ 324s 3s/step - accuracy: 0.9262 - loss: 0.2122 - precision: 0.9390 - recall: 0.9117 - val_accuracy: 0.9227 - val_loss: 0.2022 - val_precision: 0.9924 - val_recall: 0.9028 - lear

... (4 outputs left out)

97/97 ━━━━━━━━━━━━━━━━━━━━ 342s 3s/step - accuracy: 0.7658 - loss: 0.9742 - precision_2: 0.7680 - recall_2: 0.7616 - val_accuracy: 0.7430 - val_loss: 0.5593 - val_precision_2: 0.7430 - val_recall_2: 1.0000 - learning_rate: 1.0000e-04
Epoch 2/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8659 - loss: 0.3285 - precision_2: 0.8741 - recall_2: 0.8554
Epoch 2: val_accuracy did not improve from 0.74297
97/97 ━━━━━━━━━━━━━━━━━━━━ 328s 3s/step - accuracy: 0.8814 - loss: 0.3012 - precision_2: 0.8910 - recall_2: 0.8692 - val_accuracy: 0.7430 - val_loss: 0.4915 - val_precision_2: 0.7430 - val_recall_2: 1.0000 - learning_rate: 1.0000e-04
Epoch 3/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8904 - loss: 0.2986 - precision_2: 0.8914 - recall_2: 0.8898
Epoch 3: val_accuracy did not improve from 0.74297
97/97 ━━━━━━━━━━━━━━━━━━━━ 316s 3s/step - accuracy: 0.9079 - loss: 0.2585 - precision_2: 0.9121 - recall_2: 0.9027 - val_accuracy: 0.7430 - val_loss: 0.5186 - val_precision_2: 0.7430

97/97 ━━━━━━━━━━━━━━━━━━━━ 270s 3s/step - accuracy: 0.9091 - loss: 0.2436 - precision_2: 0.9189 - recall_2: 0.8976 - val_accuracy: 0.7634 - val_loss: 0.3603 - val_precision_2: 0.7585 - val_recall_2: 1.0000 - learning_rate: 1.0000e-04
Epoch 5/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9187 - loss: 0.2220 - precision_2: 0.9333 - recall_2: 0.9022
Epoch 5: val_accuracy improved from 0.76343 to 0.86637, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_kfoldc_3.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 179s 2s/step - accuracy: 0.9214 - loss: 0.2163 - precision_2: 0.9314 - recall_2: 0.9098 - val_accuracy: 0.8664 - val_loss: 0.2480 - val_precision_2: 0.8496 - val_recall_2: 0.9966 - learning_rate: 1.0000e-04
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9286 - loss: 0.1916 - precision_2: 0.9408 - recall_2: 0.9150
Epoch 6: val_accuracy improved from 0.86637 to 0.94309, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_kfoldc_3.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 161s 2s/step - accuracy: 0.9220 - loss: 0.2043 - precision_2: 0.9304 - recall_2: 0.9124 - val_accuracy: 0.9431 - val_loss: 0.1613 - val_precision_2: 0.9387 - val_recall_2: 0.9880 - learning_rate: 1.0000e-04
Epoch 7/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9433 - loss: 0.1576 - precision_2: 0.9513 - recall_2: 0.9343
Epoch 7: val_accuracy did not improve from 0.94309
97/97 ━━━━━━━━━━━━━━━━━━━━ 165s 2s/step - accuracy: 0.9381 - loss: 0.1721 - precision_2: 0.9491 - recall_2: 0.9259 - val_accuracy: 0.9405 - val_loss: 0.1729 - val_precision_2: 0.9944 - val_recall_2: 0.9251 - learning_rate: 1.0000e-04
Epoch 8/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9209 - loss: 0.2120 - precision_2: 0.9242 - recall_2: 0.9173
Epoch 8: val_accuracy did not improve from 0.94309
97/97 ━━━━━━━━━━━━━━━━━━━━ 149s 2s/step - accuracy: 0.9278 - loss: 0.2015 - precision_2: 0.9346 - recall_2: 0.9201 - val_accuracy: 0.9393 - val_loss: 0.1740 - val_precision_2: 0.9944

In [17]:
print("end")

end
time: 15 ms (started: 2026-10-03 00:23:23 +05:30)
